# Final 01 — Feature selection com ydata-profiling

Parte das **15 features candidatas** (`02_classes_bl` → `build_features_risco`) e aplica os filtros
usando os alertas do ydata, **só no treino**:

1. **ydata antes** (`reports/final/features_antes.html`): profile das 15 candidatas.
2. **Prevalência**: binária presente em < 5% do treino sai (bate com o alerta *Imbalance*). 
3. **Correlação**: enquanto o ydata acusar *High correlation* (matriz `auto`, limiar 0,5), sai a
   feature com **mais ligações** de correlação alta; empate = maior soma das correlações. A cada
   remoção o profile é refeito.
4. **ydata depois** (`reports/final/features_depois.html`): profile das que sobraram.

Alertas que **não** levam a remoção: *Duplicates* (genomas diferentes com o mesmo perfil, são
isolados reais) e *Zeros* (zero = gene ausente, não dado faltante).

In [1]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")


def encontrar_raiz_repo(marcadores=(".git", "data")):
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if all((pasta / m).exists() for m in marcadores):
            return pasta
    raise FileNotFoundError(f"Nao achei {marcadores} subindo a partir de {Path.cwd()}")


RAIZ = encontrar_raiz_repo()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

In [2]:
from ydata_profiling import ProfileReport

## 1. Dados

Treino de `df_estruturada/df_train_risco.csv` (split 70/20/10 estratificado por espécie do
`03_feature_selection`). Validação e teste não entram em nenhuma decisão.

In [3]:
pasta = RAIZ / "data/processed/df_estruturada"
df_train = pd.read_csv(pasta / "df_train_risco.csv", index_col=0)

CANDIDATAS = [c for c in df_train.columns if c != "Species"]
binarias = [c for c in CANDIDATAS if set(df_train[c].unique()) <= {0, 1}]

pasta_reports = RAIZ / "reports/final"
pasta_reports.mkdir(parents=True, exist_ok=True)
pasta_saida = RAIZ / "data/processed/final"
pasta_saida.mkdir(parents=True, exist_ok=True)

print(f"treino: {len(df_train)} genomas | {len(CANDIDATAS)} candidatas ({len(binarias)} binárias)")
df_train[CANDIDATAS].describe().T.round(2)

treino: 1558 genomas | 15 candidatas (9 binárias)


,count,mean,std,min,25%,50%,75%,max
has_carbapenemase,1558.0,0.64,0.48,0.0,0.00,1.00,1.00,1.0
has_mbl,1558.0,0.12,0.32,0.0,0.00,0.00,0.00,1.0
has_carbapenemase_serina,1558.0,0.57,0.50,0.0,0.00,1.00,1.00,1.0
n_carbapenemases,1558.0,0.69,0.55,0.0,0.00,1.00,1.00,2.0
has_esbl,1558.0,0.48,0.50,0.0,0.00,0.00,1.00,1.0
has_ampc_adquirida,1558.0,0.04,0.19,0.0,0.00,0.00,0.00,1.0
n_bl_adquiridas,1558.0,2.17,1.45,0.0,1.00,2.00,3.00,7.0
has_16s_metilase,1558.0,0.14,0.34,0.0,0.00,0.00,0.00,1.0
n_ame,1558.0,2.66,1.79,0.0,1.00,3.00,4.00,8.0
has_mcr,1558.0,0.03,0.16,0.0,0.00,0.00,0.00,1.0


## 2. ydata antes

`alertas()` lê os alertas do profile: tipo, coluna e, no caso de *High correlation*, com quais
colunas.

In [4]:
def alertas(profile):
    linhas = []
    for a in profile.get_description().alerts:
        campos = a.values.get("fields", []) if a.alert_type.name == "HIGH_CORRELATION" else []
        linhas.append({"alerta": a.alert_type.name, "coluna": a.column_name, "correlacionada_com": campos})
    return pd.DataFrame(linhas)


profile_antes = ProfileReport(df_train[CANDIDATAS], title="Features candidatas -- antes da seleção (treino)",
                              progress_bar=False)
profile_antes.to_file(pasta_reports / "features_antes.html")

alertas_antes = alertas(profile_antes)
print(alertas_antes["alerta"].value_counts().to_string())
alertas_antes


  0%|          | 0/15 [00:00<?, ?it/s]


100%|██████████| 15/15 [00:00<00:00, 284.38it/s]

alerta
HIGH_CORRELATION    11
ZEROS                5
IMBALANCE            2
DUPLICATES           1


,alerta,coluna,correlacionada_com
0,DUPLICATES,None,[]
1,HIGH_CORRELATION,carbapenemase_plasmidial,"[has_carbapenemase, has_carbapenemase_serina, n_bl_adquiridas, n_carbapenemases]"
2,HIGH_CORRELATION,has_16s_metilase,"[n_ame, n_genes_adquiridos]"
3,HIGH_CORRELATION,has_carbapenemase,"[carbapenemase_plasmidial, has_carbapenemase_serina, n_bl_adquiridas, n_carbapenemases]"
4,HIGH_CORRELATION,has_carbapenemase_serina,"[carbapenemase_plasmidial, has_carbapenemase, n_carbapenemases]"
5,HIGH_CORRELATION,has_esbl,"[n_bl_adquiridas, n_genes_adquiridos]"
6,HIGH_CORRELATION,has_mbl,[n_carbapenemases]
7,HIGH_CORRELATION,n_ame,"[has_16s_metilase, n_bl_adquiridas, n_classes_adquiridas, n_genes_adquiridos]"
8,HIGH_CORRELATION,n_bl_adquiridas,"[carbapenemase_plasmidial, has_carbapenemase, has_esbl, n_ame, n_classes_adquiridas, n_genes_adquiridos]"
9,HIGH_CORRELATION,n_carbapenemases,"[carbapenemase_plasmidial, has_carbapenemase, has_carbapenemase_serina, has_mbl]"


### 2.1 Matriz de correlação do ydata (`auto`)

O ydata escolhe o coeficiente pelo tipo do par (Spearman entre numéricas, Cramér's V entre
categóricas, …) e acusa *High correlation* acima de **0,5**.

In [5]:
corr_antes = profile_antes.get_description().correlations["auto"]
LIMIAR = profile_antes.config.correlations["auto"].threshold
print(f"limiar do ydata: {LIMIAR}")

plt.figure(figsize=(11, 9))
sns.heatmap(corr_antes, annot=True, fmt=".2f", cmap="rocket_r", vmin=0, vmax=1)
plt.title(f"Correlação ydata (auto) -- treino, 15 candidatas (limiar {LIMIAR})")
plt.tight_layout()
plt.show()

limiar do ydata: 0.5


## 3. Filtro de prevalência (< 5%)

In [6]:
PREV_MIN = 0.05
prevalencia = (df_train[binarias] > 0).mean().round(3)
raras = prevalencia[prevalencia < PREV_MIN].index.tolist()
imbalance = alertas_antes.query("alerta == 'IMBALANCE'")["coluna"].tolist()

print(f"prevalência < {PREV_MIN:.0%}: {raras}")
print(f"alerta Imbalance do ydata: {imbalance}")
prevalencia.sort_values().to_frame("prevalência no treino")

prevalência < 5%: ['has_ampc_adquirida', 'has_mcr']
alerta Imbalance do ydata: ['has_ampc_adquirida', 'has_mcr']


,prevalência no treino
has_mcr,0.025
has_ampc_adquirida,0.036
has_mbl,0.116
has_16s_metilase,0.137
has_pmqr,0.236
carbapenemase_plasmidial,0.481
has_esbl,0.485
has_carbapenemase_serina,0.567
has_carbapenemase,0.644


## 4. Filtro de correlação (iterativo)

A cada rodada: refaz o profile com as features que sobraram, conta quantas ligações de *High
correlation* cada uma tem e remove a de maior grau (empate: maior soma das correlações). Para
quando o ydata não acusar mais nenhuma correlação alta.

Uma feature que se correlaciona com muitas outras é, em geral, um **agregado** delas (ex.:
`n_genes_adquiridos` soma as outras contagens), então ela carrega pouca informação própria.

In [ ]:
restantes = [c for c in CANDIDATAS if c not in raras]
rodadas = []
while True:
    profile = ProfileReport(df_train[restantes], progress_bar=False)
    corr = profile.get_description().correlations["auto"]
    ligacoes = alertas(profile).query("alerta == 'HIGH_CORRELATION'")
    if ligacoes.empty:
        break
    ligacoes = ligacoes.assign(
        #grau = quantas colunas ela está correlacionada
        grau=ligacoes["correlacionada_com"].map(len),
        #soma_corr = soma dos |r| com essas parceiras
        soma_corr=[corr.loc[c, v].abs().sum() for c, v in zip(ligacoes["coluna"], ligacoes["correlacionada_com"])],
    ).sort_values(["grau", "soma_corr"], ascending=False)
    sai = ligacoes.iloc[0]
    # a com correlação mais alta sai
    rodadas.append({"rodada": len(rodadas) + 1, 
                    "removida": sai["coluna"], 
                    "grau": sai["grau"],
                    "soma_corr": round(sai["soma_corr"], 2),
                    "correlacionada_com": ", ".join(sai["correlacionada_com"])})
    restantes.remove(sai["coluna"])

rodadas = pd.DataFrame(rodadas)
FEATURES_FINAIS = restantes
print("FEATURES_FINAIS:", FEATURES_FINAIS)
rodadas


  0%|          | 0/13 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:00<00:00, 431.33it/s]


  0%|          | 0/12 [00:00<?, ?it/s]


100%|██████████| 12/12 [00:00<00:00, 291.28it/s]


  0%|          | 0/11 [00:00<?, ?it/s]

...


100%|██████████| 8/8 [00:00<00:00, 304.54it/s]


  0%|          | 0/7 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 477.00it/s]

FEATURES_FINAIS: ['has_mbl', 'has_carbapenemase_serina', 'has_esbl', 'has_16s_metilase', 'has_pmqr', 'n_classes_adquiridas', 'pct_adquiridos_plasmidial']


,rodada,removida,grau,soma_corr,correlacionada_com
0,1,n_bl_adquiridas,6,3.69,"carbapenemase_plasmidial, has_carbapenemase, has_esbl, n_ame, n_classes_adquiridas, n_genes_adquiridos"
1,2,n_carbapenemases,4,3.11,"carbapenemase_plasmidial, has_carbapenemase, has_carbapenemase_serina, has_mbl"
2,3,n_genes_adquiridos,4,2.80,"has_16s_metilase, has_esbl, n_ame, n_classes_adquiridas"
3,4,has_carbapenemase,2,1.56,"carbapenemase_plasmidial, has_carbapenemase_serina"
4,5,n_ame,2,1.30,"has_16s_metilase, n_classes_adquiridas"
5,6,carbapenemase_plasmidial,1,0.60,has_carbapenemase_serina


## 5. ydata depois

In [8]:
profile_depois = ProfileReport(df_train[FEATURES_FINAIS], title="Features finais -- depois da seleção (treino)",
                               progress_bar=False)
profile_depois.to_file(pasta_reports / "features_depois.html")

alertas_depois = alertas(profile_depois)
alertas_depois


  0%|          | 0/7 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 341.39it/s]

,alerta,coluna,correlacionada_com
0,DUPLICATES,None,[]
1,ZEROS,n_classes_adquiridas,[]
2,ZEROS,pct_adquiridos_plasmidial,[]


In [9]:
corr_depois = profile_depois.get_description().correlations["auto"]
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr_depois, annot=True, fmt=".2f", cmap="rocket_r", vmin=0, vmax=1)
plt.title("Correlação ydata (auto) -- features finais")
plt.tight_layout()
plt.show()

print(f"perfis distintos com as features finais: {len(df_train[FEATURES_FINAIS].drop_duplicates())} "
      f"em {len(df_train)} genomas")

perfis distintos com as features finais: 685 em 1558 genomas


## 6. Resumo da seleção (15 → finais)

In [10]:
motivo = {c: f"prevalência {prevalencia[c]:.1%} < 5%" for c in raras}
motivo.update({r.removida: f"correlação alta (rodada {r.rodada}): {r.correlacionada_com}" for r in rodadas.itertuples()})
selecao = pd.DataFrame({
    "tipo": ["binária" if c in binarias else "contagem/proporção" for c in CANDIDATAS],
    "decisao": ["fica" if c in FEATURES_FINAIS else "sai" for c in CANDIDATAS],
    "motivo": [motivo.get(c, "sem alerta de correlação alta") for c in CANDIDATAS],
}, index=pd.Index(CANDIDATAS, name="feature")).sort_values("decisao")
selecao

,tipo,decisao,motivo
feature,,,
has_mbl,binária,fica,sem alerta de correlação alta
has_carbapenemase_serina,binária,fica,sem alerta de correlação alta
has_esbl,binária,fica,sem alerta de correlação alta
has_16s_metilase,binária,fica,sem alerta de correlação alta
has_pmqr,binária,fica,sem alerta de correlação alta
n_classes_adquiridas,contagem/proporção,fica,sem alerta de correlação alta
pct_adquiridos_plasmidial,contagem/proporção,fica,sem alerta de correlação alta
has_carbapenemase,binária,sai,"correlação alta (rodada 4): carbapenemase_plasmidial, has_carbapenemase_serina"
n_carbapenemases,contagem/proporção,sai,"correlação alta (rodada 2): carbapenemase_plasmidial, has_carbapenemase, has_carbapenemase_serina, has_mbl"


## 7. Salvando

In [11]:
DESCRITIVAS = ["has_carbapenemase", "carbapenemase_plasmidial", "n_bl_adquiridas", "n_genes_adquiridos",
               "has_ampc_adquirida", "has_mcr"]
saida = {
    "features_finais": FEATURES_FINAIS,
    "descritivas": DESCRITIVAS,
    "criterio": f"treino; binária com prevalência < {PREV_MIN:.0%} sai; remoção iterativa pelo alerta "
                f"High correlation do ydata (auto, limiar {LIMIAR}): sai a feature de maior grau, "
                "empate = maior soma das correlações",
}
(pasta_saida / "features_selecionadas.json").write_text(json.dumps(saida, indent=2, ensure_ascii=False))
selecao.to_csv(pasta_saida / "selecao_features.csv")
saida

{'features_finais': ['has_mbl',
  'has_carbapenemase_serina',
  'has_esbl',
  'has_16s_metilase',
  'has_pmqr',
  'n_classes_adquiridas',
  'pct_adquiridos_plasmidial'],
 'descritivas': ['has_carbapenemase',
  'carbapenemase_plasmidial',
  'n_bl_adquiridas',
  'n_genes_adquiridos',
  'has_ampc_adquirida',
  'has_mcr'],
 'criterio': 'treino; binária com prevalência < 5% sai; remoção iterativa pelo alerta High correlation do ydata (auto, limiar 0.5): sai a feature de maior grau, empate = maior soma das correlações'}

### Exportação para o app

Tabela da seleção, rodadas de correlação e as duas matrizes do ydata, lidas pela página Clustering.

In [12]:
pasta_app = RAIZ / "app/data/clustering"
pasta_app.mkdir(parents=True, exist_ok=True)
(pasta_app / "features_selecionadas.json").write_text(json.dumps(saida, indent=2, ensure_ascii=False))
selecao.to_csv(pasta_app / "selecao_features.csv")
rodadas.to_csv(pasta_app / "rodadas_correlacao.csv", index=False)
corr_antes.round(3).to_csv(pasta_app / "corr_ydata_antes.csv")
corr_depois.round(3).to_csv(pasta_app / "corr_ydata_depois.csv")